## Paths and core config

In [61]:
from pathlib import Path
ROOT = Path().resolve().parent  # Scripts/ → repo root

BASE_PATH = ROOT / "Model_Outputs"

DATA_PATH           = ROOT / "Data_Files"   / "stroke_master_final_cleaned.parquet"
COLS_TO_DROP_PATH   = ROOT / "Stat_Outputs" / "cols_to_drop.csv"
NUM_FEAT_PATH       = ROOT / "Stat_Outputs" / "num_feat.csv"
REMOVED_FEATURES_PATH = ROOT / "Stat_Outputs" / "removed_features.csv"

#CF_FEATURES = ["eProcedures_03_ID", "eResponse_07", "eProcedures_10_ID", "eDisposition_32"]
CF_FEATURES = ["eResponse_07", "eProcedures_10_ID", "eDisposition_32"]

MIN_COUNT_CF = 50
TOP_K_CF = 20

## Imports

In [62]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, ParameterGrid
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.base import clone

from sklearn.metrics import (
    matthews_corrcoef,
    accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    make_scorer
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

from xgboost import XGBClassifier

## General helpers, metrics, data loading, preprocessing, model builders

In [63]:
# ----------------------------
# Config helpers
# ----------------------------
def get_thresh_grid(n=99):
    return np.linspace(0.01, 0.99, n)


def ensure_outdir(outdir):
    outdir = os.path.abspath(outdir)
    os.makedirs(outdir, exist_ok=True)
    return outdir


# ----------------------------
# Metric helpers
# ----------------------------
def metric_name_normalize(metric):
    metric = str(metric).strip().lower()
    if metric in {"mcc"}:
        return "mcc"
    if metric in {"macro_f1", "macro-f1", "macrof1"}:
        return "macro_f1"
    if metric in {"f1_0", "f10", "f1-0"}:
        return "f1_0"
    if metric in {"f1_1", "f11", "f1-1"}:
        return "f1_1"
    raise ValueError("metric must be 'mcc', 'macro_f1', 'f1_0', or 'f1_1'.")


def _safe_average_precision(y_true_bin, p_pos):
    y_true_bin = np.asarray(y_true_bin).astype(int)
    if len(np.unique(y_true_bin)) < 2:
        return np.nan
    return float(average_precision_score(y_true_bin, p_pos))


def score_at_threshold(y_true, p, threshold):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p, dtype=float)
    yhat = (p >= float(threshold)).astype(int)

    cm = confusion_matrix(y_true, yhat, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel().tolist()

    f1_1 = float(f1_score(y_true, yhat, pos_label=1, average="binary", zero_division=0))
    f1_0 = float(f1_score(y_true, yhat, pos_label=0, average="binary", zero_division=0))
    f1_gap = float(abs(f1_0 - f1_1))

    pr_auc_1 = _safe_average_precision((y_true == 1).astype(int), p)
    pr_auc_0 = _safe_average_precision((y_true == 0).astype(int), 1.0 - p)

    if np.isnan(pr_auc_0) and np.isnan(pr_auc_1):
        pr_auc_macro = np.nan
    elif np.isnan(pr_auc_0):
        pr_auc_macro = pr_auc_1
    elif np.isnan(pr_auc_1):
        pr_auc_macro = pr_auc_0
    else:
        pr_auc_macro = float((pr_auc_0 + pr_auc_1) / 2.0)

    out = {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, yhat)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, yhat)),
        "precision": float(precision_score(y_true, yhat, zero_division=0)),
        "recall": float(recall_score(y_true, yhat, zero_division=0)),
        "f1": float(f1_1),
        "f1_0": f1_0,
        "f1_1": f1_1,
        "abs_f1_gap": f1_gap,
        "macro_f1": float(f1_score(y_true, yhat, average="macro", zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, yhat)),
        "roc_auc": float(roc_auc_score(y_true, p)) if len(np.unique(y_true)) == 2 else np.nan,
        "pr_auc": pr_auc_1,
        "pr_auc_0": pr_auc_0,
        "pr_auc_1": pr_auc_1,
        "pr_auc_macro": pr_auc_macro,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }
    return out


def get_prob_or_score(model, X_in):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_in)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X_in)
        s = (s - np.min(s)) / (np.max(s) - np.min(s) + 1e-12)
        return s
    raise ValueError("Model is lacking predict_proba and decision_function.")


def pick_threshold(y_true, p, metric, thresh_grid):
    metric = metric_name_normalize(metric)

    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p, dtype=float)

    best_t = 0.5
    best_val = -np.inf

    for t in thresh_grid:
        yhat = (p >= float(t)).astype(int)
        if len(np.unique(yhat)) < 2:
            continue

        if metric == "mcc":
            val = matthews_corrcoef(y_true, yhat)
        elif metric == "macro_f1":
            val = f1_score(y_true, yhat, average="macro", zero_division=0)
        elif metric == "f1_0":
            val = f1_score(y_true, yhat, pos_label=0, average="binary", zero_division=0)
        elif metric == "f1_1":
            val = f1_score(y_true, yhat, pos_label=1, average="binary", zero_division=0)
        else:
            raise ValueError("Unsupported metric in pick_threshold")

        if val > best_val:
            best_val = val
            best_t = float(t)

    return float(best_t), float(best_val)


def prefix_dict(d, prefix):
    return {f"{prefix}_{k}": v for k, v in d.items()}


# ----------------------------
# Data loading (NO IMPUTATION)
# ----------------------------
def load_complete_case_dataset(
    data_path,
    cols_to_drop_path,
    num_feat_path,
    removed_features_path,
    target_col="EMSSceneTimeMin",
    key_col="PcrKey",
    y_col="scene_time_binary",
    threshold_minutes=10,
):
    df = pd.read_parquet(data_path)

    cols_to_drop = pd.read_csv(cols_to_drop_path)["column_name"].astype(str).tolist()
    num_feat_original = pd.read_csv(num_feat_path)["column_name"].astype(str).tolist()
    removed = pd.read_csv(removed_features_path).iloc[:, 0].astype(str).tolist()

    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")
    df = df.drop(columns=[c for c in removed if c in df.columns], errors="ignore")

    num_feat = [c for c in num_feat_original if c in df.columns and c != target_col]
    cols_to_exclude = set(num_feat) | {key_col, target_col}
    cat_feat = [c for c in df.columns if c not in cols_to_exclude]

    for c in num_feat + [target_col]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=[target_col]).copy()
    df[y_col] = (df[target_col] > float(threshold_minutes)).astype(int)

    df = df.replace({pd.NA: np.nan})

    feature_cols = cat_feat + num_feat
    df_cc = df.dropna(subset=feature_cols + [y_col]).copy()

    X = df_cc[feature_cols].copy()
    y = df_cc[y_col].astype(int).copy()

    meta = {
        "cat_feat": cat_feat,
        "num_feat": num_feat,
        "feature_cols": feature_cols,
        "target_col": target_col,
        "y_col": y_col,
    }
    return X, y, meta


def build_preprocessor(cat_feat, num_feat):
    prep = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feat),
            ("num", StandardScaler(), num_feat),
        ],
        remainder="drop",
    )
    return prep


# ----------------------------
# Models, workflows, grids
# ----------------------------
def build_base_estimators(random_state, xgb_n_jobs=1):
    xgb_kwargs = {
        "random_state": random_state,
        "n_jobs": int(xgb_n_jobs),
        "eval_metric": "logloss",
        "tree_method": "hist",
    }

    return {
        "Logistic Regression": LogisticRegression(
            random_state=random_state, max_iter=2000, solver="lbfgs", n_jobs=-1
        ),
        "Decision Tree": DecisionTreeClassifier(random_state=random_state),
        "Random Forest": RandomForestClassifier(random_state=random_state, n_jobs=-1),
        "XGBoost": XGBClassifier(**xgb_kwargs),
    }


def build_weighted_estimators(y_train, random_state, xgb_n_jobs=1):
    classes = np.array([0, 1])
    cw = compute_class_weight(class_weight="balanced", classes=classes, y=np.asarray(y_train))
    class_weight = {0: float(cw[0]), 1: float(cw[1])}

    n_pos = int((np.asarray(y_train) == 1).sum())
    n_neg = int((np.asarray(y_train) == 0).sum())
    scale_pos_weight = float(n_neg / max(n_pos, 1))

    xgb_kwargs = {
        "random_state": random_state,
        "n_jobs": int(xgb_n_jobs),
        "eval_metric": "logloss",
        "scale_pos_weight": scale_pos_weight,
        "tree_method": "hist",
    }

    return {
        "Logistic Regression": LogisticRegression(
            random_state=random_state, max_iter=2000, solver="lbfgs",
            n_jobs=-1, class_weight=class_weight
        ),
        "Decision Tree": DecisionTreeClassifier(
            random_state=random_state, class_weight=class_weight
        ),
        "Random Forest": RandomForestClassifier(
            random_state=random_state, n_jobs=-1, class_weight=class_weight
        ),
        "XGBoost": XGBClassifier(**xgb_kwargs),
    }


def build_pipeline(workflow, prep, clf, random_state):
    if workflow in {"unbalanced", "weighted"}:
        return SkPipeline([("prep", prep), ("clf", clf)])

    if workflow == "undersampled":
        return ImbPipeline([
            ("prep", prep),
            ("rus", RandomUnderSampler(random_state=random_state)),
            ("clf", clf),
        ])

    raise ValueError("workflow must be: unbalanced, weighted, undersampled.")


def get_param_grids():
    return {
        "Logistic Regression": {"clf__C": [0.01, 0.1, 1, 10]},
        "Decision Tree": {
            "clf__max_depth": [5, 10, 15, None],
            "clf__min_samples_split": [2, 5, 10],
        },
        "Random Forest": {
            "clf__n_estimators": [100, 200],
            "clf__max_depth": [10, 15, 20],
        },
        "XGBoost": {
            "clf__n_estimators": [100, 200],
            "clf__max_depth": [3, 6, 10],
            "clf__learning_rate": [0.01, 0.1, 0.2],
        },
    }


def get_cv_scorer(metric):
    metric = metric_name_normalize(metric)
    if metric == "mcc":
        return make_scorer(matthews_corrcoef)
    if metric == "macro_f1":
        return make_scorer(f1_score, average="macro", zero_division=0)
    if metric == "f1_0":
        return make_scorer(f1_score, pos_label=0, average="binary", zero_division=0)
    if metric == "f1_1":
        return make_scorer(f1_score, pos_label=1, average="binary", zero_division=0)
    raise ValueError("metric must be one of: mcc, macro_f1, f1_0, f1_1")


# ----------------------------
# Candidate selection
# ----------------------------
def select_best_by_valid_generalization(cand_df, metric, eps):
    metric = metric_name_normalize(metric)
    valid_key = f"valid_tuned_{metric}"

    best_valid = float(cand_df[valid_key].max())
    pool = cand_df[cand_df[valid_key] >= (best_valid - float(eps))].copy()
    if pool.empty:
        pool = cand_df.copy()

    best_row = pool.sort_values(
        by=[
            "gen_gap_metric",
            valid_key,
            "valid_tuned_pr_auc_macro",
            "best_cv_metric",
        ],
        ascending=[True, False, False, False],
    ).iloc[0]

    return best_row

## Counterfactual toolkit

In [64]:
# ============================================================
# Stage 3 Counterfactual Toolkit (reuses scene_time_rates)
# ============================================================

def scene_time_rates(
    df: pd.DataFrame,
    features,
    target: str = "scene_time_binary",
    min_count: int = 1,
    dropna: bool = True,
    sort_by: str = "prob_grt_10m",
    ascending: bool = False
) -> pd.DataFrame:
    if isinstance(features, str):
        features = [features]

    cols = list(features) + [target]
    work = df[cols].copy()

    if dropna:
        work = work.dropna(subset=cols)

    out = (
        work.groupby(features, dropna=not dropna)[target]
            .agg(count="size", prob_grt_10m="mean")
            .reset_index()
    )

    out = out[out["count"] >= min_count].copy()

    if sort_by == "prob_grt_10m":
        out = out.sort_values(["prob_grt_10m", "count"], ascending=[ascending, False])
    elif sort_by == "count":
        out = out.sort_values(["count", "prob_grt_10m"], ascending=[False, ascending])
    else:
        raise ValueError("sort_by must be 'prob_grt_10m' or 'count'")

    return out.reset_index(drop=True)


def lowest_prob_combinations_with_dict(
    df: pd.DataFrame,
    features,
    target_col: str = "scene_time_binary",
    top_k: int = 20,
    min_count: int = 50,
    dropna: bool = True
):
    if isinstance(features, str):
        features = [features]

    ranked = scene_time_rates(
        df=df,
        features=features,
        target=target_col,
        min_count=min_count,
        dropna=dropna,
        sort_by="prob_grt_10m",
        ascending=True
    )

    if ranked.empty:
        raise ValueError(
            f"No combinations meet min_count={min_count}. "
            f"Try lowering min_count or check missingness."
        )

    ranked = ranked.head(top_k).reset_index(drop=True)

    best_row = ranked.iloc[0]
    replacement_dict = {feat: str(best_row[feat]) for feat in features}

    return replacement_dict, ranked


def predict_binary_from_proba(p: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    return (p >= threshold).astype(int)


def _as_cat_string(val):
    sval = str(val)
    if sval.endswith(".0"):
        sval = sval[:-2]
    return sval


def counterfactual_replace(
    model,
    X: pd.DataFrame,
    replacement_dict: dict,
    threshold: float = 0.5,
    target: str = "pred1",
    cast_object: bool = True
):
    X_base = X.copy()

    p_base = model.predict_proba(X_base)[:, 1]
    yhat_base = predict_binary_from_proba(p_base, threshold)

    if target == "pred1":
        mask = (yhat_base == 1)
    elif target == "all":
        mask = np.ones(len(X_base), dtype=bool)
    else:
        raise ValueError("target must be 'pred1' or 'all'")

    X_cf = X_base.copy()

    cleaned_repl = {}
    for col, val in replacement_dict.items():
        if col not in X_cf.columns:
            raise KeyError(f"Column not found in X: {col}")

        sval = _as_cat_string(val)
        cleaned_repl[col] = sval

        if cast_object:
            X_cf[col] = X_cf[col].astype("object")

        X_cf.loc[mask, col] = sval

    p_cf = model.predict_proba(X_cf)[:, 1]
    yhat_cf = predict_binary_from_proba(p_cf, threshold)

    intervened_n = int(mask.sum())
    base_rate_1 = float((yhat_base == 1).mean())
    cf_rate_1 = float((yhat_cf == 1).mean())
    delta_rate_1 = float(cf_rate_1 - base_rate_1)

    if intervened_n > 0:
        yb_mask = yhat_base[mask]
        yc_mask = yhat_cf[mask]

        baseline_pred_1_count = int((yb_mask == 1).sum())
        baseline_pred_0_count = int((yb_mask == 0).sum())

        cf_pred_1_count = int((yc_mask == 1).sum())
        cf_pred_0_count = int((yc_mask == 0).sum())

        flip_1to0 = int(((yb_mask == 1) & (yc_mask == 0)).sum())
        flip_0to1 = int(((yb_mask == 0) & (yc_mask == 1)).sum())

        flip_1to0_efficiency = (
            float(flip_1to0 / baseline_pred_1_count)
            if baseline_pred_1_count > 0 else np.nan
        )
        flip_0to1_harm_rate = (
            float(flip_0to1 / baseline_pred_0_count)
            if baseline_pred_0_count > 0 else np.nan
        )
    else:
        baseline_pred_1_count = 0
        baseline_pred_0_count = 0
        cf_pred_1_count = 0
        cf_pred_0_count = 0
        flip_1to0 = 0
        flip_0to1 = 0
        flip_1to0_efficiency = np.nan
        flip_0to1_harm_rate = np.nan

    relative_change_pred_rate_1 = (
        float(delta_rate_1 / base_rate_1)
        if base_rate_1 > 0 else np.nan
    )

    summary = {
        "replacement_dict": cleaned_repl,
        "threshold": float(threshold),
        "target": target,
        "intervened_rows": intervened_n,
        "baseline_pred_rate_1": base_rate_1,
        "cf_pred_rate_1": cf_rate_1,
        "delta_pred_rate_1": delta_rate_1,
        "relative_change_pred_rate_1": relative_change_pred_rate_1,
        "baseline_pred_1_count": baseline_pred_1_count,
        "baseline_pred_0_count": baseline_pred_0_count,
        "cf_pred_1_count": cf_pred_1_count,
        "cf_pred_0_count": cf_pred_0_count,
        "flip_1to0_efficiency": flip_1to0_efficiency,
        "flip_0to1_harm_rate": flip_0to1_harm_rate,
    }

    baseline_df = pd.DataFrame({"p_base": p_base, "yhat_base": yhat_base}, index=X.index)
    cf_df = pd.DataFrame({"p_cf": p_cf, "yhat_cf": yhat_cf}, index=X.index)

    return summary, baseline_df, cf_df, X_cf


def run_counterfactual_from_train(
    model,
    train_df: pd.DataFrame,
    X_test: pd.DataFrame,
    features,
    target_col: str = "scene_time_binary",
    min_count: int = 50,
    top_k: int = 20,
    threshold: float = 0.5,
    target: str = "pred1",
    dropna: bool = True
):
    replacement_dict, ranked_table = lowest_prob_combinations_with_dict(
        df=train_df,
        features=features,
        target_col=target_col,
        top_k=top_k,
        min_count=min_count,
        dropna=dropna
    )

    summary, baseline_df, cf_df, X_cf = counterfactual_replace(
        model=model,
        X=X_test,
        replacement_dict=replacement_dict,
        threshold=threshold,
        target=target
    )

    return summary, replacement_dict, ranked_table, baseline_df, cf_df, X_cf

## Universal complete-case load and TVT split

In [65]:
X, y, meta = load_complete_case_dataset(
    data_path=DATA_PATH,
    cols_to_drop_path=COLS_TO_DROP_PATH,
    num_feat_path=NUM_FEAT_PATH,
    removed_features_path=REMOVED_FEATURES_PATH,
    target_col="EMSSceneTimeMin",
    key_col="PcrKey",
    y_col="scene_time_binary",
    threshold_minutes=10,
)

print("Complete-case cohort:", X.shape, "positive rate:", round(float(y.mean()), 6))

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Train:", X_train.shape, "Valid:", X_valid.shape, "Test:", X_test.shape)

train_df_with_target = X_train.copy()
train_df_with_target["scene_time_binary"] = y_train.values

Complete-case cohort: (160566, 31) positive rate: 0.868067
Train: (128452, 31) Valid: (16057, 31) Test: (16057, 31)


## Main TVT pipeline - macro F1 tuned

In [66]:
def run_tvt_pipeline_macrof1(
    X_train,
    y_train,
    X_valid,
    y_valid,
    X_test,
    y_test,
    train_df_with_target,
    meta,
    metric="macro_f1",
    eps=0.02,
    random_state=42,
    n_splits=5,
    thresh_grid=None,
    outdir=None,
):
    metric = metric_name_normalize(metric)
    if metric != "macro_f1":
        raise ValueError("This function is intended for macro_f1 tuning only.")

    if thresh_grid is None:
        thresh_grid = get_thresh_grid(99)
    if outdir is None:
        outdir = os.path.join(BASE_PATH, "out_tvt_macro_f1")
    ensure_outdir(outdir)

    prep = build_preprocessor(meta["cat_feat"], meta["num_feat"])

    base_models = build_base_estimators(random_state)
    weighted_models = build_weighted_estimators(y_train, random_state)

    param_grids = get_param_grids()
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    scorer = get_cv_scorer(metric)

    workflows = [
        ("weighted", weighted_models),
        # ("unbalanced", base_models),
        # ("undersampled", base_models),
    ]

    candidates = []

    for workflow_name, model_dict in workflows:
        for model_name, clf in model_dict.items():
            pipe_base = build_pipeline(workflow_name, prep, clf, random_state)

            best_params = None
            best_cv_metric = -np.inf

            for params in ParameterGrid(param_grids[model_name]):
                pipe = clone(pipe_base)
                pipe.set_params(**params)

                fold_scores = []

                for tr_idx, va_idx in cv.split(X_train, y_train):
                    X_tr = X_train.iloc[tr_idx]
                    y_tr = y_train.iloc[tr_idx]
                    X_va = X_train.iloc[va_idx]
                    y_va = y_train.iloc[va_idx]

                    pipe_fold = clone(pipe)
                    pipe_fold.fit(X_tr, y_tr)

                    p_va = get_prob_or_score(pipe_fold, X_va)
                    _, best_fold_metric = pick_threshold(
                        y_true=y_va.to_numpy(),
                        p=p_va,
                        metric=metric,
                        thresh_grid=thresh_grid,
                    )
                    fold_scores.append(best_fold_metric)

                mean_cv_metric = float(np.mean(fold_scores)) if len(fold_scores) > 0 else np.nan

                print(
                    "   Params:", params,
                    "| inner_cv_macro_f1:", round(float(mean_cv_metric), 6) if not np.isnan(mean_cv_metric) else "nan"
                )

                if not np.isnan(mean_cv_metric) and mean_cv_metric > best_cv_metric:
                    best_cv_metric = mean_cv_metric
                    best_params = params

            if best_params is None:
                print("Skipping", workflow_name, model_name, "because no valid macro_f1-based params were found.")
                continue

            best_est = clone(pipe_base)
            best_est.set_params(**best_params)
            best_est.fit(X_train, y_train)

            p_train = get_prob_or_score(best_est, X_train)
            p_valid = get_prob_or_score(best_est, X_valid)

            best_t, best_valid_metric_val = pick_threshold(
                y_true=y_valid.to_numpy(),
                p=p_valid,
                metric=metric,
                thresh_grid=thresh_grid,
            )

            train_tuned = score_at_threshold(y_train, p_train, threshold=best_t)
            valid_tuned = score_at_threshold(y_valid, p_valid, threshold=best_t)

            gen_gap_metric = float(abs(train_tuned[metric] - valid_tuned[metric]))
            gen_gap_pr_auc = float(abs(train_tuned["pr_auc_macro"] - valid_tuned["pr_auc_macro"]))

            cand = {
                "workflow": workflow_name,
                "model": model_name,
                "best_params": best_params,
                "best_cv_metric": float(best_cv_metric),
                "best_threshold_valid": float(best_t),
                "best_valid_metric_val": float(best_valid_metric_val),
                "gen_gap_metric": gen_gap_metric,
                "gen_gap_pr_auc": gen_gap_pr_auc,
                **prefix_dict(train_tuned, "train_tuned"),
                **prefix_dict(valid_tuned, "valid_tuned"),
                "_best_estimator": best_est,
            }
            cand["candidate_id"] = len(candidates)
            candidates.append(cand)

            print(
                "Done:", workflow_name, "|", model_name,
                "| inner_cv_macro_f1:", round(float(best_cv_metric), 6),
                "| best_t:", round(best_t, 4),
                "| valid macro_f1:", round(float(valid_tuned["macro_f1"]), 6),
                "| valid f1_0:", round(float(valid_tuned["f1_0"]), 6),
                "| valid f1_1:", round(float(valid_tuned["f1_1"]), 6),
                "| valid pr_auc_macro:", round(float(valid_tuned["pr_auc_macro"]), 6),
                "| metric_gap(train-valid):", round(float(gen_gap_metric), 6),
            )

    cand_df = pd.DataFrame([{k: v for k, v in c.items() if not k.startswith("_")} for c in candidates])

    candidates_csv = os.path.join(outdir, "candidates_valid_only.csv")
    cand_df.to_csv(candidates_csv, index=False)
    print("Saved:", candidates_csv)

    best_row = select_best_by_valid_generalization(cand_df, metric=metric, eps=eps)
    best_id = int(best_row["candidate_id"])
    best_est = candidates[best_id]["_best_estimator"]
    best_t = float(best_row["best_threshold_valid"])

    print("Selected best:")
    print("  workflow:", best_row["workflow"])
    print("  model   :", best_row["model"])
    print("  valid_macro_f1:", round(float(best_row["valid_tuned_macro_f1"]), 6))
    print("  gap:", round(float(best_row["gen_gap_metric"]), 6))
    print("  best_t:", round(best_t, 6))

    p_test = get_prob_or_score(best_est, X_test)
    test_default = score_at_threshold(y_test, p_test, threshold=0.5)
    test_tuned = score_at_threshold(y_test, p_test, threshold=best_t)

    final = best_row.to_dict()
    final.update(prefix_dict(test_default, "test_default"))
    final.update(prefix_dict(test_tuned, "test_tuned"))
    final["selected_metric"] = metric
    final["eps"] = float(eps)

    best_csv = os.path.join(outdir, "best_model_final_test.csv")
    pd.DataFrame([final]).to_csv(best_csv, index=False)
    print("Saved:", best_csv)

    return (
        cand_df,
        pd.DataFrame([final]),
        best_est,
        float(best_t),
    )

In [67]:
cand_macro_f1, best_macro_f1, tvt_pipe_macro_f1, tvt_t_macro_f1 = run_tvt_pipeline_macrof1(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    X_test=X_test,
    y_test=y_test,
    train_df_with_target=train_df_with_target,
    meta=meta,
    metric="macro_f1",
    eps=0.01,
)

best_macro_f1

   Params: {'clf__C': 0.01} | inner_cv_macro_f1: 0.668824
   Params: {'clf__C': 0.1} | inner_cv_macro_f1: 0.668929
   Params: {'clf__C': 1} | inner_cv_macro_f1: 0.66474
   Params: {'clf__C': 10} | inner_cv_macro_f1: 0.657514
Done: weighted | Logistic Regression | inner_cv_macro_f1: 0.668929 | best_t: 0.33 | valid macro_f1: 0.669199 | valid f1_0: 0.437862 | valid f1_1: 0.900535 | valid pr_auc_macro: 0.678924 | metric_gap(train-valid): 0.012942
   Params: {'clf__max_depth': 5, 'clf__min_samples_split': 2} | inner_cv_macro_f1: 0.619501
   Params: {'clf__max_depth': 5, 'clf__min_samples_split': 5} | inner_cv_macro_f1: 0.619501
   Params: {'clf__max_depth': 5, 'clf__min_samples_split': 10} | inner_cv_macro_f1: 0.619487
   Params: {'clf__max_depth': 10, 'clf__min_samples_split': 2} | inner_cv_macro_f1: 0.641172
   Params: {'clf__max_depth': 10, 'clf__min_samples_split': 5} | inner_cv_macro_f1: 0.641354
   Params: {'clf__max_depth': 10, 'clf__min_samples_split': 10} | inner_cv_macro_f1: 0.641

,workflow,model,best_params,best_cv_metric,best_threshold_valid,best_valid_metric_val,gen_gap_metric,gen_gap_pr_auc,train_tuned_threshold,train_tuned_accuracy,...,test_tuned_pr_auc,test_tuned_pr_auc_0,test_tuned_pr_auc_1,test_tuned_pr_auc_macro,test_tuned_tn,test_tuned_fp,test_tuned_fn,test_tuned_tp,selected_metric,eps
0,weighted,Logistic Regression,{'clf__C': 0.1},0.668929,0.33,0.669199,0.012942,0.018269,0.33,0.837496,...,0.960871,0.407568,0.960871,0.684219,1050,1069,1650,12288,macro_f1,0.01


## Feature importance for the selected macro F1 tuned winner

In [68]:
from sklearn.compose import ColumnTransformer

def _get_feature_names_from_preprocessor(prep: ColumnTransformer):
    return prep.get_feature_names_out()

def extract_feature_importance_full(fitted_pipe) -> pd.DataFrame:
    prep = fitted_pipe.named_steps["prep"]
    clf = fitted_pipe.named_steps["clf"]

    feat_names = _get_feature_names_from_preprocessor(prep)

    if hasattr(clf, "feature_importances_"):
        imp = np.asarray(clf.feature_importances_, dtype=float)

    elif hasattr(clf, "coef_"):
        coef = np.asarray(clf.coef_, dtype=float)

        if coef.ndim == 2:
            if coef.shape[0] == 1:
                imp = np.abs(coef[0])
            else:
                imp = np.mean(np.abs(coef), axis=0)
        else:
            imp = np.abs(coef)

    else:
        raise ValueError(
            "This model does not expose feature_importances_ or coef_."
        )

    if len(feat_names) != len(imp):
        raise RuntimeError(f"Mismatch: {len(feat_names)} names vs {len(imp)} importances.")

    fi_all = (
        pd.DataFrame({"feature": feat_names, "importance": imp})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    return fi_all


def collapse_ohe_sum_single_model_using_lists(
    fi_long: pd.DataFrame,
    cat_feat: list,
    num_feat: list,
    top_n: int = 50
) -> pd.DataFrame:
    cat_parents_sorted = sorted(cat_feat, key=len, reverse=True)

    def strip_prefix(s: str) -> str:
        return s.split("__", 1)[-1]

    def parent_from_cat_raw(raw: str) -> str:
        for parent in cat_parents_sorted:
            if raw == parent or raw.startswith(parent + "_"):
                return parent
        return raw.split("_", 1)[0] if "_" in raw else raw

    def parent_feature_name(encoded_name: str) -> str:
        if pd.isna(encoded_name):
            return encoded_name

        s = str(encoded_name)
        raw = strip_prefix(s)

        if s.startswith("num__") or raw in num_feat:
            return raw

        if s.startswith("cat__"):
            return parent_from_cat_raw(raw)

        if raw in num_feat:
            return raw
        if raw in cat_feat:
            return raw

        return parent_from_cat_raw(raw)

    tmp = fi_long[["feature", "importance"]].dropna().copy()
    tmp["parent_feature"] = tmp["feature"].apply(parent_feature_name)

    collapsed = (
        tmp.groupby("parent_feature", as_index=False)["importance"]
           .sum()
           .sort_values("importance", ascending=False)
           .head(int(top_n))
           .reset_index(drop=True)
           .rename(columns={"parent_feature": "feature"})
    )

    return collapsed


cat_feat = meta["cat_feat"]
num_feat = meta["num_feat"]

fi_all_macro_f1 = extract_feature_importance_full(tvt_pipe_macro_f1)

collapsed_all_macro_f1 = collapse_ohe_sum_single_model_using_lists(
    fi_long=fi_all_macro_f1,
    cat_feat=cat_feat,
    num_feat=num_feat,
    top_n=50
)

collapsed_all_macro_f1

,feature,importance
0,eProcedures_03_ID,658.717174
1,eSituation_11_ID,84.348715
2,eSituation_09_ID,69.471798
3,eScene_09_ID,33.323711
4,eProcedures_10_ID,22.356951
5,eResponse_24_ID,21.346310
6,eDisposition_20_ID,18.312555
7,eVitals_20_ID,6.148013
8,eResponse_07,3.828289
9,ePatient_14_ID,3.442096


## Final policy simulation / counterfactual analysis on macro F1 tuned winner

In [69]:
train_df_cf = X_train.copy()
train_df_cf["scene_time_binary"] = y_train.values

feats = CF_FEATURES

summary_macro_f1, repl_macro_f1, ranked_macro_f1, base_df_macro_f1, cf_df_macro_f1, X_cf_macro_f1 = run_counterfactual_from_train(
    model=tvt_pipe_macro_f1,
    train_df=train_df_cf,
    X_test=X_test,
    features=feats,
    target_col="scene_time_binary",
    min_count=MIN_COUNT_CF,
    top_k=TOP_K_CF,
    threshold=tvt_t_macro_f1,
    target="all",
    dropna=True,
)

summary_macro_f1

{'replacement_dict': {'eResponse_07': '2207015',
  'eProcedures_10_ID': '3413',
  'eDisposition_32': '4232005'},
 'threshold': 0.33,
 'target': 'all',
 'intervened_rows': 16057,
 'baseline_pred_rate_1': 0.8318490378028274,
 'cf_pred_rate_1': 0.8786199165473002,
 'delta_pred_rate_1': 0.04677087874447283,
 'relative_change_pred_rate_1': 0.05622520026952162,
 'baseline_pred_1_count': 13357,
 'baseline_pred_0_count': 2700,
 'cf_pred_1_count': 14108,
 'cf_pred_0_count': 1949,
 'flip_1to0_efficiency': 0.010406528412068579,
 'flip_0to1_harm_rate': 0.3296296296296296}

In [72]:
# ── Global result as DataFrame ───────────────────────────────────────────────
global_cf_policy_results_df = pd.DataFrame([{
    "target": "all",
    "train_size": len(train_df_cf),
    "test_size": len(X_test),
    **summary_macro_f1
}])


# ── Decode lookup (same pattern as subgroup) ─────────────────────────────────
LOOKUP_DIR = ROOT / "Data_Files" / "2024_lookups_parquet"

cf_global_coded_vals = {}
for row_dict in global_cf_policy_results_df["replacement_dict"]:
    for feat, val in row_dict.items():
        cf_global_coded_vals.setdefault(feat, set()).add(val)

def build_decode_mapping(feat, coded_vals):
    base     = feat[:-3] if feat.endswith("_ID") else feat
    lkp_path = LOOKUP_DIR / f"lookup_{base}.parquet"
    if not lkp_path.exists():
        return {v: v for v in coded_vals if pd.notna(v)}
    lkp = pd.read_parquet(lkp_path)
    return {v: lkp.loc[lkp["enc_id"] == int(v), "combo_key"].iloc[0]
            for v in coded_vals if pd.notna(v)}

global_decode_maps = {feat: build_decode_mapping(feat, vals)
                      for feat, vals in cf_global_coded_vals.items()}

df_global_decoded = global_cf_policy_results_df.copy()
df_global_decoded["replacement_dict"] = df_global_decoded["replacement_dict"].apply(
    lambda d: {feat: global_decode_maps.get(feat, {}).get(val, val) for feat, val in d.items()}
)

df_global_decoded

,target,train_size,test_size,replacement_dict,threshold,intervened_rows,baseline_pred_rate_1,cf_pred_rate_1,delta_pred_rate_1,relative_change_pred_rate_1,baseline_pred_1_count,baseline_pred_0_count,cf_pred_1_count,cf_pred_0_count,flip_1to0_efficiency,flip_0to1_harm_rate
0,all,128452,16057,"{'eResponse_07': '2207015', 'eProcedures_10_ID...",0.33,16057,0.831849,0.87862,0.046771,0.056225,13357,2700,14108,1949,0.010407,0.32963


## Subgroup policy changes on macro F1 tuned winner

In [70]:
subgroup_features = [
    "Urbanicity_ID"
]

subgroup_values_df = (
    train_df_cf[subgroup_features]
    .dropna()
    .drop_duplicates()
)

subgroup_cf_policy_results_macro_f1 = []

for _, row in subgroup_values_df.iterrows():

    train_mask = np.ones(len(train_df_cf), dtype=bool)
    test_mask = np.ones(len(X_test), dtype=bool)

    for col in subgroup_features:
        train_mask &= (train_df_cf[col] == row[col])
        test_mask &= (X_test[col] == row[col])

    train_subset = train_df_cf[train_mask]
    test_subset = X_test[test_mask]

    if len(train_subset) < MIN_COUNT_CF or len(test_subset) == 0:
        continue

    try:
        summary_sub_macro_f1, repl_sub_macro_f1, ranked_sub_macro_f1, base_df_sub_macro_f1, cf_df_sub_macro_f1, X_cf_sub_macro_f1 = run_counterfactual_from_train(
            model=tvt_pipe_macro_f1,
            train_df=train_subset,
            X_test=test_subset,
            features=feats,
            target_col="scene_time_binary",
            min_count=MIN_COUNT_CF,
            top_k=TOP_K_CF,
            threshold=tvt_t_macro_f1,
            target="all",
            dropna=True,
        )

        subgroup_cf_policy_results_macro_f1.append({
            **{col: row[col] for col in subgroup_features},
            "train_size": len(train_subset),
            "test_size": len(test_subset),
            **summary_sub_macro_f1
        })

    except ValueError:
        continue

subgroup_cf_policy_results_macro_f1_df = pd.DataFrame(subgroup_cf_policy_results_macro_f1)
subgroup_cf_policy_results_macro_f1_df

,Urbanicity_ID,train_size,test_size,replacement_dict,threshold,target,intervened_rows,baseline_pred_rate_1,cf_pred_rate_1,delta_pred_rate_1,relative_change_pred_rate_1,baseline_pred_1_count,baseline_pred_0_count,cf_pred_1_count,cf_pred_0_count,flip_1to0_efficiency,flip_0to1_harm_rate
0,4904.0,105680,13221,"{'eResponse_07': '2207015', 'eProcedures_10_ID...",0.33,all,13221,0.827245,0.878148,0.050904,0.061534,10937,2284,11610,1611,0.007223,0.329247
1,4901.0,2195,282,"{'eResponse_07': '2207017', 'eProcedures_10_ID...",0.33,all,282,0.755319,0.702128,-0.053191,-0.070423,213,69,198,84,0.089202,0.057971
2,4903.0,8961,1157,"{'eResponse_07': '2207017', 'eProcedures_10_ID...",0.33,all,1157,0.883319,0.909248,0.025929,0.029354,1022,135,1052,105,0.020548,0.377778
3,4902.0,11616,1397,"{'eResponse_07': '2207017', 'eProcedures_10_ID...",0.33,all,1397,0.848246,0.805297,-0.042949,-0.050633,1185,212,1125,272,0.054008,0.018868


In [71]:
LOOKUP_DIR = ROOT / "Data_Files" / "2024_lookups_parquet"

# ── 1. Collect all coded values that need decoding ──────────────────────────
coded_vals_by_feat = {col: set(subgroup_cf_policy_results_macro_f1_df[col].dropna().tolist())
                     for col in subgroup_features}

for row_dict in subgroup_cf_policy_results_macro_f1_df["replacement_dict"]:
    for feat, val in row_dict.items():
        coded_vals_by_feat.setdefault(feat, set()).add(val)

# ── 2. Build decode mapping for each feature ────────────────────────────────
def build_decode_mapping(feat, coded_vals):
    base     = feat[:-3] if feat.endswith("_ID") else feat
    lkp_path = LOOKUP_DIR / f"lookup_{base}.parquet"
    if not lkp_path.exists():
        return {v: v for v in coded_vals if pd.notna(v)}
    lkp = pd.read_parquet(lkp_path)
    return {v: lkp.loc[lkp["enc_id"] == int(v), "combo_key"].iloc[0]
            for v in coded_vals if pd.notna(v)}

decode_maps = {feat: build_decode_mapping(feat, vals)
               for feat, vals in coded_vals_by_feat.items()}

# ── 3. Apply to dataframe ────────────────────────────────────────────────────
df_decoded = subgroup_cf_policy_results_macro_f1_df.copy()

for col in subgroup_features:
    df_decoded[col] = df_decoded[col].map(decode_maps[col])

df_decoded["replacement_dict"] = df_decoded["replacement_dict"].apply(
    lambda d: {feat: decode_maps.get(feat, {}).get(val, val) for feat, val in d.items()}
)

df_decoded


,Urbanicity_ID,train_size,test_size,replacement_dict,threshold,target,intervened_rows,baseline_pred_rate_1,cf_pred_rate_1,delta_pred_rate_1,relative_change_pred_rate_1,baseline_pred_1_count,baseline_pred_0_count,cf_pred_1_count,cf_pred_0_count,flip_1to0_efficiency,flip_0to1_harm_rate
0,Urban,105680,13221,"{'eResponse_07': '2207015', 'eProcedures_10_ID...",0.33,all,13221,0.827245,0.878148,0.050904,0.061534,10937,2284,11610,1611,0.007223,0.329247
1,Frontier,2195,282,"{'eResponse_07': '2207017', 'eProcedures_10_ID...",0.33,all,282,0.755319,0.702128,-0.053191,-0.070423,213,69,198,84,0.089202,0.057971
2,Suburban,8961,1157,"{'eResponse_07': '2207017', 'eProcedures_10_ID...",0.33,all,1157,0.883319,0.909248,0.025929,0.029354,1022,135,1052,105,0.020548,0.377778
3,Rural,11616,1397,"{'eResponse_07': '2207017', 'eProcedures_10_ID...",0.33,all,1397,0.848246,0.805297,-0.042949,-0.050633,1185,212,1125,272,0.054008,0.018868
